# Coupon Experiment Decision Framework — Walkthrough

This notebook walks through the framework stage by stage using simulated data.

Note: the dataset contains hidden latent fields for later validation. Stages 1-4 only use observable fields; latent truth and long-term outcomes are revealed in Stage 5.

In [ ]:
import pandas as pd

df = pd.read_csv("../data/simulated_coupon_experiment.csv")

# Stages 1-4 only use observable fields. The CSV also contains latent ground
# truth and long-term outcome columns, which are intentionally not loaded or
# shown until Stage 5.
observable_cols = [
    "user_id",
    "group",
    "coupon_received",
    "signup_day",
    "exposure_day",
    "made_first_purchase_30d",
    "days_to_first_purchase",
    "repeat_visits_14d",
    "browse_sessions_14d",
    "product_views_14d",
    "cart_adds_14d",
    "discount_page_views_14d",
    "coupon_used",
]

df[observable_cols].head()

## Stage 1: Measurement Design

Stage 1 is about designing the experiment so it can answer a stronger question than "did the 30-day first-purchase rate go up?" — it should also capture what *kind* of value any lift represents. That means committing up front to the primary metric, guardrail metrics, holdout logic, observation windows, and the post-period tracking needed to later assess incremental demand, customer quality, and borrowed-demand risk.

### Trap 1: The Headline Lift

In [ ]:
rates = (
    df.groupby("group")["made_first_purchase_30d"]
    .mean()
    .rename("first_purchase_rate_30d")
)

control_rate = rates.loc["control"]
treatment_rate = rates.loc["treatment"]
lift_pp = (treatment_rate - control_rate) * 100

print(f"Control   30d first-purchase rate: {control_rate:.2%}")
print(f"Treatment 30d first-purchase rate: {treatment_rate:.2%}")
print(f"Headline lift:                     {lift_pp:+.2f}pp")

rates.to_frame()

The coupon appears to improve the 30-day first-purchase rate. But this only answers whether the rate went up. It does not tell us what kind of value that lift represents.

### Trap 2: Same Lift, Different Reality

| Scenario | Headline lift | Dominant hidden reality | Decision implication |
|----------|---------------|-------------------------|----------------------|
| A | +6pp | More persuadable high-value users | Consider scale |
| B | +6pp | More discount-driven and pulled-forward users | Adjust or stop |

Two experiments can show the same headline lift while representing very different underlying realities. The same +6pp could justify scaling in one case and stopping in another. This is the gap the rest of the framework aims to close. We will return to validate this with long-term outcomes in Stage 5.